# Notebook 3 — Mask2Former Swin-Tiny (transformer)

**Model:** Mask2Former Swin-Tiny dari Hugging Face `transformers`, pretrained COCO instance
(`facebook/mask2former-swin-tiny-coco-instance`).
**Keluarga:** transformer berbasis *query*. 100 query belajar masing-masing memprediksi satu kelas dan satu mask
(resolusi 1/4 input) lewat *masked attention*. Tidak ada box, NMS, atau RoI; pasangan prediksi–GT ditentukan
dengan Hungarian matching.

Resep (mengikuti konfigurasi resmi Mask2Former, disesuaikan untuk T4): AdamW lr 1e-4 (backbone ×0.1), weight decay
0.05, jadwal *poly*, gradient clipping 0.01, multi-scale training (sisi panjang 0.75–1× ukuran input), horizontal flip.
Forward memakai fp16, tetapi loss dihitung dalam fp32 karena matcher bawaan tidak stabil di fp16.
Validasi setiap `EVAL_EVERY` epoch; bobot dengan mask AP validasi tertinggi disimpan di `weights/best/`.

Kalau muncul *CUDA out of memory*, turunkan `BATCH` menjadi 2.

**Cara menjalankan di Kaggle:**
1. *Settings* → *Accelerator*: **GPU T4 x2**, *Internet*: **On**.
2. *Add Input* → *Notebooks* → pilih output **Notebook 0** (berisi `indoor_seg_data.zip`).
3. **Pilot dulu** (`PILOT = True`, jalankan interaktif): 2 epoch pada data 100% untuk mengukur waktu per epoch.
   Notebook mencetak saran `EPOCHS` agar run 100% selesai dalam `TARGET_HOURS_100` jam.
4. Isi `EPOCHS`, set `PILOT = False`, lalu **Save Version → Save & Run All** (berjalan di background, maks 12 jam).

**Cara kerja dengan 2 GPU:** setiap run memakai satu GPU dan berjalan sebagai proses terpisah. GPU0 menjalankan run 100%,
GPU1 menjalankan run 25% lalu 50%. Batch size, learning rate, dan jumlah epoch identik untuk ketiga fraksi, jadi
satu-satunya yang berbeda antar run adalah jumlah data latih.

**Evaluasi:** semua model dievaluasi dengan fungsi pycocotools yang sama (`seg_common.py`, identik di Notebook 1–3)
pada split `valid`, `test`, dan `test_own` (kalau sudah ada), maksimal 100 prediksi per gambar.

**Output** (`/kaggle/working`): `runs/<model>_<fraksi>/` (bobot, `results.csv`, prediksi, evaluasi),
`metrics_<model>.csv`, `per_class_<model>.csv`, `gpu_time_log.csv`, dan `logs/`.

**Test set buatan sendiri yang datang belakangan:** jalankan ulang Notebook 0 dengan `OWN_TEST_*` terisi, lalu
jalankan notebook ini dengan `RUN_TRAINING = False` dan output versi training notebook ini ditambahkan sebagai input.
Bobot lama dipakai, tidak ada training ulang.

In [1]:
!pip install -q pycocotools

## 1. Konfigurasi

In [2]:
import glob
import json
import os
import shutil
import subprocess
import sys
import threading
import time
import zipfile
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
import yaml

MODEL_KEY = "mask2former_swin_t"
MODEL_LABEL = "Mask2Former Swin-T"
FAMILY = "transformer"
PRETRAINED = "facebook/mask2former-swin-tiny-coco-instance"

EPOCHS = 60          # pilot: 2 epoch = 4,4 mnt -> saran 54; train murni 93,7 dtk/epoch + eval tiap 2 epoch
                     # -> ~1,8 mnt/epoch, jadi 60 epoch ~1,8 jam (target 2 jam)
BATCH = 4            # per GPU (satu run = satu GPU); turunkan ke 2 kalau OOM
LR = 1e-4
IMGSZ = 640
WORKERS = 2
EVAL_EVERY = 2

TAGS = ["025", "050", "100"]          # fraksi data latih (NRP digit 3)
SEED = 42
MAX_HOURS_PER_RUN = 3.0               # pengaman: run berhenti rapi setelah epoch yang melewati batas ini

PILOT = False                         # True: 2 epoch pada data 100% untuk mengukur waktu per epoch (sudah dijalankan)
PILOT_EPOCHS = 2
TARGET_HOURS_100 = 2.0                # target durasi run 100% untuk saran jumlah epoch
RUN_TRAINING = True                   # False: lewati training, pakai bobot dari output versi sebelumnya

DATA_ZIP_GLOB = "/kaggle/input/**/indoor_seg_data.zip"
PREV_RUNS_GLOB = "/kaggle/input/**/runs"
DATA_ROOT = Path("/tmp/data")
WORK = Path("/kaggle/working")
RUNS = WORK / "runs"
POLL_SECONDS = 120                    # interval laporan progres

## 2. Data dari Notebook 0

In [3]:
zips = sorted(glob.glob(DATA_ZIP_GLOB, recursive=True))
assert zips, "indoor_seg_data.zip tidak ditemukan. Tambahkan output Notebook 0 lewat Add Input."
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)
with zipfile.ZipFile(zips[0]) as z:
    z.extractall(DATA_ROOT)

# path absolut di YAML YOLO disamakan dengan lokasi ekstrak
for y in DATA_ROOT.glob("data_*.yaml"):
    cfg = yaml.safe_load(y.read_text(encoding="utf-8"))
    cfg["path"] = DATA_ROOT.as_posix()
    y.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")

INFO = json.loads((DATA_ROOT / "dataset_info.json").read_text(encoding="utf-8"))
EVAL_SPLITS = [s for s in ["valid", "test", "test_own"] if (DATA_ROOT / "annotations" / f"{s}.json").exists()]
print("zip    :", zips[0])
print("kelas  :", [c["name"] for c in INFO["classes"]])
print("subset :", INFO["subsets"])
print("evaluasi pada:", EVAL_SPLITS)
if "test_own" not in EVAL_SPLITS:
    print("-> test_own belum ada. Setelah Notebook 0 dijalankan ulang dengan test set buatanmu, "
          "jalankan notebook ini dengan RUN_TRAINING = False.")

zip    : /kaggle/input/notebooks/sultandaulay/cv-is-data-prep/indoor_seg_data.zip
kelas  : ['backpack', 'bed', 'blanket', 'book', 'bottle', 'bowl', 'box', 'cabinet-merged', 'cardboard', 'ceiling-merged', 'cell phone', 'chair', 'clock', 'couch', 'counter', 'cup', 'curtain', 'dining table', 'door-stuff', 'fire-extinguisher', 'floor-other-merged', 'floor-wood', 'flower', 'food-other-merged', 'fork', 'hair drier', 'handbag', 'keyboard', 'knife', 'laptop', 'light', 'microwave', 'mirror-stuff', 'mouse', 'oven', 'pan', 'paper', 'paper-merged', 'person', 'pillow', 'plate', 'plug', 'potted plant', 'refrigerator', 'remote', 'rug-merged', 'scissor', 'scissors', 'shelf', 'sink', 'spoon', 'st-', 'stairs', 'stove-merged', 'table-merged', 'teddy bear', 'toaster', 'toilet', 'toothbrush', 'towel', 'toy', 'trash-bin', 'tree-merged', 'tv', 'vase', 'wall', 'wall-brick', 'wall-other-merged', 'wall-stone', 'wall-tile', 'wall-wood', 'wallet', 'window-blind', 'window-other']
subset : {'025': {'images': 99, 'i

## 3. Kode training dan evaluasi

Ditulis ke file supaya setiap run berjalan sebagai proses terpisah di GPU-nya sendiri. `seg_common.py` identik di Notebook 1–3.

In [4]:
%%writefile seg_common.py
"""Fungsi bersama ketiga notebook training: seed, encode mask, evaluasi COCO (pycocotools), log CSV."""
import contextlib
import csv
import io
import json
import random
from pathlib import Path

import numpy as np
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

STATS = ["AP", "AP50", "AP75", "APs", "APm", "APl"]


def set_seed(seed):
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def read_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def write_json(obj, path, indent=None):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=indent)


def append_csv(path, row, fields):
    path = Path(path)
    new = not path.exists()
    with open(path, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore", restval="")
        if new:
            w.writeheader()
        w.writerow(row)


def encode_mask(mask):
    rle = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))
    rle["counts"] = rle["counts"].decode("ascii")
    return rle


def det(image_id, category_id, mask, score, bbox=None):
    """Satu prediksi dalam format hasil COCO. bbox (x, y, w, h) dihitung dari mask kalau tidak diberikan."""
    rle = encode_mask(mask)
    if bbox is None:
        bbox = mask_utils.toBbox(rle).tolist()
    return {"image_id": int(image_id), "category_id": int(category_id), "segmentation": rle,
            "bbox": [float(v) for v in bbox], "score": float(score)}


def coco_evaluate(gt_json, dets):
    """Mask AP dan box AP gaya COCO, plus mask AP per kelas.
    Untuk 'segm', bbox dibuang dari prediksi supaya area dihitung dari mask (sama seperti Detectron2)."""
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO(str(gt_json))
    metrics, per_class = {}, {}
    for iou_type in ["segm", "bbox"]:
        if not dets:
            metrics.update({f"{iou_type}_{s}": 0.0 for s in STATS})
            continue
        d = [{k: v for k, v in x.items() if not (iou_type == "segm" and k == "bbox")} for x in dets]
        with contextlib.redirect_stdout(io.StringIO()):
            ev = COCOeval(gt, gt.loadRes(d), iou_type)
            ev.evaluate()
            ev.accumulate()
            ev.summarize()
        metrics.update({f"{iou_type}_{s}": float(v) for s, v in zip(STATS, ev.stats[:6])})
        if iou_type == "segm":
            prec = ev.eval["precision"][:, :, :, 0, -1]   # IoU x recall x kelas, area=all, maxDets=100
            for k, cat_id in enumerate(ev.params.catIds):
                p = prec[:, :, k]
                p = p[p > -1]
                per_class[gt.cats[cat_id]["name"]] = float(p.mean()) if p.size else float("nan")
    return metrics, per_class


def evaluate_and_save(gt_json, dets, out_dir, split, extra=None):
    metrics, per_class = coco_evaluate(gt_json, dets)
    write_json(dets, Path(out_dir) / f"preds_{split}.json")
    result = {"split": split, **(extra or {}), **metrics, "per_class_segm_AP": per_class}
    write_json(result, Path(out_dir) / f"eval_{split}.json", indent=2)
    print(f"[{split}] mask AP={metrics['segm_AP']:.4f}  AP50={metrics['segm_AP50']:.4f}  "
          f"box AP={metrics['bbox_AP']:.4f}  ({len(dets)} prediksi)", flush=True)
    return result

Writing seg_common.py


In [5]:
%%writefile train_mask2former.py
"""Training & prediksi Mask2Former (Hugging Face transformers) untuk satu fraksi data. Satu proses, satu GPU."""
import argparse
import random
import time
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from PIL import Image
from pycocotools import mask as mask_utils
from transformers import Mask2FormerForUniversalSegmentation

from seg_common import append_csv, coco_evaluate, det, evaluate_and_save, read_json, set_seed, write_json

warnings.filterwarnings("ignore", message="Detected call of")   # GradScaler melewati step pertama (normal di fp16)

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]     # ImageNet, sama dengan Mask2Former Swin
FIELDS = ["epoch", "lr", "loss", "loss_cross_entropy", "loss_mask", "loss_dice", "skipped_iters", "train_time_s",
          "val_segm_AP", "val_segm_AP50", "val_bbox_AP", "saved_best", "elapsed_h", "gpu_mem_gb"]


def parse():
    p = argparse.ArgumentParser()
    p.add_argument("mode", choices=["train", "predict"])
    p.add_argument("--tag", required=True)
    p.add_argument("--name", required=True)
    p.add_argument("--data-root", required=True)
    p.add_argument("--out", required=True)
    p.add_argument("--weights", default="facebook/mask2former-swin-tiny-coco-instance")
    p.add_argument("--epochs", type=int, default=40)
    p.add_argument("--batch", type=int, default=4)
    p.add_argument("--lr", type=float, default=1e-4)
    p.add_argument("--backbone-lr-mult", type=float, default=0.1)
    p.add_argument("--wd", type=float, default=0.05)
    p.add_argument("--clip", type=float, default=0.01)                 # seperti konfigurasi resmi Mask2Former
    p.add_argument("--imgsz", type=int, default=640)
    p.add_argument("--workers", type=int, default=2)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--eval-every", type=int, default=2)
    p.add_argument("--max-hours", type=float, default=0)
    p.add_argument("--amp", action=argparse.BooleanOptionalAction, default=True)
    p.add_argument("--splits", nargs="+", default=["valid", "test"])
    return p.parse_args()


def load_image(path, long_side):
    """Resize agar sisi panjang = long_side (rasio dijaga), lalu normalisasi."""
    img = Image.open(path).convert("RGB")
    w, h = img.size
    s = long_side / max(h, w)
    nh, nw = max(1, round(h * s)), max(1, round(w * s))
    x = TF.normalize(TF.to_tensor(img.resize((nw, nh), Image.BILINEAR)), MEAN, STD)
    return x, (h, w), (nh, nw)


def pad_to(x, size):
    out = x.new_zeros((*x.shape[:-2], size, size))
    out[..., : x.shape[-2], : x.shape[-1]] = x
    return out


class TrainSet(torch.utils.data.Dataset):
    def __init__(self, json_path, img_dir, imgsz):
        coco = read_json(json_path)
        anns = defaultdict(list)
        for a in coco["annotations"]:
            anns[a["image_id"]].append(a)
        self.items = [(im, anns[im["id"]]) for im in coco["images"] if anns[im["id"]]]
        self.img_dir, self.imgsz = Path(img_dir), imgsz
        self.scales = sorted(set(range(int(imgsz * 0.75) // 32 * 32, imgsz + 1, 32)) | {imgsz})

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        im, anns = self.items[i]
        x, (h, w), (nh, nw) = load_image(self.img_dir / im["file_name"], random.choice(self.scales))
        masks = np.stack([mask_utils.decode(mask_utils.merge(mask_utils.frPyObjects(a["segmentation"], h, w)))
                          for a in anns])
        masks = F.interpolate(torch.from_numpy(masks)[None].float(), size=(nh, nw), mode="nearest")[0].to(torch.uint8)
        labels = torch.tensor([a["category_id"] - 1 for a in anns], dtype=torch.int64)   # 0..N-1
        if random.random() < 0.5:
            x, masks = x.flip(-1), masks.flip(-1)
        keep = masks.flatten(1).amax(1) > 0
        pixel_mask = torch.zeros((self.imgsz, self.imgsz), dtype=torch.long)
        pixel_mask[:nh, :nw] = 1
        return pad_to(x, self.imgsz), pixel_mask, pad_to(masks[keep], self.imgsz), labels[keep]


def collate_train(batch):
    batch = [b for b in batch if len(b[3])]
    if not batch:
        return None
    return (torch.stack([b[0] for b in batch]), torch.stack([b[1] for b in batch]),
            [b[2] for b in batch], [b[3] for b in batch])


def load_model(src, class_names=None):
    kw = {}
    if class_names:
        kw = dict(id2label=dict(enumerate(class_names)), label2id={n: i for i, n in enumerate(class_names)},
                  ignore_mismatched_sizes=True)
    return Mask2FormerForUniversalSegmentation.from_pretrained(src, **kw)


@torch.no_grad()
def instance_inference(class_logits, mask_logits, valid_hw, orig_hw, padded, topk=100):
    """Inferensi instance Mask2Former: top-k pasangan (query, kelas), skor = prob kelas x rata-rata prob mask."""
    scores = class_logits.float().softmax(-1)[:, :-1]          # buang kelas "no object"
    num_classes = scores.shape[1]
    top_scores, idx = scores.flatten().topk(min(topk, scores.numel()))
    q, cls = idx // num_classes, idx % num_classes
    m = F.interpolate(mask_logits[q][None].float(), size=(padded, padded), mode="bilinear", align_corners=False)[0]
    m = m[:, : valid_hw[0], : valid_hw[1]]
    m = F.interpolate(m[None], size=orig_hw, mode="bilinear", align_corners=False)[0]
    binary = m > 0
    mask_score = (m.sigmoid() * binary).flatten(1).sum(1) / (binary.flatten(1).sum(1) + 1e-6)
    return binary, cls, top_scores * mask_score


@torch.no_grad()
def predict_split(model, root, split, dev, amp, imgsz):
    gt_json = root / "annotations" / f"{split}.json"
    images = read_json(gt_json)["images"]
    model.eval()
    dets, t_inf = [], 0.0
    for i, im in enumerate(images):
        t0 = time.time()
        x, _, (nh, nw) = load_image(root / "images" / split / im["file_name"], imgsz)
        pm = torch.zeros((1, imgsz, imgsz), dtype=torch.long, device=dev)
        pm[:, :nh, :nw] = 1
        with torch.autocast(dev.type, dtype=torch.float16, enabled=amp):
            out = model(pixel_values=pad_to(x, imgsz)[None].to(dev), pixel_mask=pm)
        binary, cls, scores = instance_inference(out.class_queries_logits[0], out.masks_queries_logits[0],
                                                 (nh, nw), (im["height"], im["width"]), imgsz)
        binary, cls, scores = binary.cpu().numpy(), cls.cpu().numpy(), scores.cpu().numpy()
        if i > 0:
            t_inf += time.time() - t0
        for m, c, s in zip(binary, cls, scores):
            if m.any():
                dets.append(det(im["id"], c + 1, m, s))
    return gt_json, dets, 1000 * t_inf / max(len(images) - 1, 1)


def train(a):
    set_seed(a.seed)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = a.amp and dev.type == "cuda"
    root, run = Path(a.data_root), Path(a.out) / a.name
    (run / "weights").mkdir(parents=True, exist_ok=True)
    (run / "results.csv").unlink(missing_ok=True)
    write_json(vars(a), run / "args.json", indent=2)
    class_names = [c["name"] for c in read_json(root / "dataset_info.json")["classes"]]

    ds = TrainSet(root / "annotations" / f"train_{a.tag}.json", root / "images" / "train", a.imgsz)
    dl = torch.utils.data.DataLoader(ds, batch_size=a.batch, shuffle=True, num_workers=a.workers,
                                     collate_fn=collate_train, generator=torch.Generator().manual_seed(a.seed),
                                     persistent_workers=a.workers > 0, pin_memory=dev.type == "cuda")
    model = load_model(a.weights, class_names).to(dev)

    groups = defaultdict(list)                   # (backbone?, tanpa weight decay?) -> parameter
    for n, p in model.named_parameters():
        if p.requires_grad:
            groups[(n.startswith("model.pixel_level_module.encoder."),
                    p.ndim == 1 or "norm" in n or "embed" in n)].append(p)
    assert any(bb for bb, _ in groups), "parameter backbone tidak ditemukan (nama modul berubah?)"
    opt = torch.optim.AdamW([{"params": ps, "lr": a.lr * (a.backbone_lr_mult if bb else 1.0),
                              "weight_decay": 0.0 if no_wd else a.wd} for (bb, no_wd), ps in groups.items()])
    total = a.epochs * len(dl)
    warm = max(1, min(100, total // 20))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda it: min(1.0, (it + 1) / warm) * (1 - min(it, total - 1) / total) ** 0.9)   # poly
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    print(f"train_{a.tag}: {len(ds)} gambar beranotasi, {len(dl)} iterasi/epoch, {total} iterasi, device={dev}, "
          f"amp={amp}", flush=True)

    best, t_start, stopped_by_time, ep = -1.0, time.time(), False, 0
    for ep in range(1, a.epochs + 1):
        model.train()
        t0, sums, n, skipped = time.time(), defaultdict(float), 0, 0
        for batch in dl:
            if batch is None:
                continue
            pv, pm, mask_labels, class_labels = batch
            pv, pm = pv.to(dev, non_blocking=True), pm.to(dev, non_blocking=True)
            mask_labels = [m.to(dev, non_blocking=True).float() for m in mask_labels]
            class_labels = [c.to(dev, non_blocking=True) for c in class_labels]
            # forward dengan fp16, tetapi loss (termasuk Hungarian matching) dihitung dalam fp32:
            # matcher bawaan memakai clamp 1e10 yang menjadi inf di fp16.
            with torch.autocast(dev.type, dtype=torch.float16, enabled=amp):
                out = model(pixel_values=pv, pixel_mask=pm, output_auxiliary_logits=True)
            aux = [{k: v.float() for k, v in d.items()} for d in (out.auxiliary_logits or [])]
            loss_dict = model.get_loss_dict(out.masks_queries_logits.float(), out.class_queries_logits.float(),
                                            mask_labels, class_labels, aux)
            loss = sum(loss_dict.values())
            opt.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):
                skipped += 1
                sched.step()
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), a.clip)
            scaler.step(opt)
            scaler.update()
            sched.step()
            n += 1
            sums["loss"] += loss.item()
            for k in ("loss_cross_entropy", "loss_mask", "loss_dice"):      # layer terakhir saja
                sums[k] += loss_dict[k].item()
        row = {"epoch": ep, "lr": f"{max(g['lr'] for g in opt.param_groups):.7f}", "skipped_iters": skipped,
               "train_time_s": round(time.time() - t0, 1), **{k: round(v / max(n, 1), 5) for k, v in sums.items()}}
        stopped_by_time = a.max_hours > 0 and (time.time() - t_start) / 3600 > a.max_hours
        if ep % a.eval_every == 0 or ep == a.epochs or stopped_by_time:
            gt_json, dets, _ = predict_split(model, root, "valid", dev, amp, a.imgsz)
            m, _ = coco_evaluate(gt_json, dets)
            row.update(val_segm_AP=round(m["segm_AP"], 5), val_segm_AP50=round(m["segm_AP50"], 5),
                       val_bbox_AP=round(m["bbox_AP"], 5))
            if m["segm_AP"] > best:
                best = m["segm_AP"]
                model.save_pretrained(run / "weights" / "best")
                row["saved_best"] = 1
        row["elapsed_h"] = round((time.time() - t_start) / 3600, 4)
        row["gpu_mem_gb"] = round(torch.cuda.max_memory_allocated() / 1e9, 2) if dev.type == "cuda" else 0
        append_csv(run / "results.csv", row, FIELDS)
        print(" | ".join(f"{k}={row[k]}" for k in FIELDS if k in row), flush=True)
        if stopped_by_time:
            print(f"[time_guard] melewati {a.max_hours} jam, berhenti setelah epoch {ep}", flush=True)
            break
    write_json({"tag": a.tag, "epochs_planned": a.epochs, "epochs_done": ep, "stopped_by_time": stopped_by_time,
                "best_val_segm_AP": round(best, 5), "train_hours": round((time.time() - t_start) / 3600, 4),
                "train_images": len(ds)}, run / "train_summary.json", indent=2)


def predict(a):
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = a.amp and dev.type == "cuda"
    root, run = Path(a.data_root), Path(a.out) / a.name
    model = load_model(run / "weights" / "best").to(dev)
    for split in a.splits:
        if not (root / "annotations" / f"{split}.json").exists():
            print(f"[{split}] tidak ada, dilewati")
            continue
        gt_json, dets, ms = predict_split(model, root, split, dev, amp, a.imgsz)
        evaluate_and_save(gt_json, dets, run, split, {"model": "mask2former", "tag": a.tag, "ms_per_image": round(ms, 2)})


if __name__ == "__main__":
    args = parse()
    train(args) if args.mode == "train" else predict(args)

Writing train_mask2former.py


## 4. Bobot pretrained dan runner

In [6]:
from transformers import Mask2FormerForUniversalSegmentation

# Unduh bobot sekali ke cache Hugging Face, supaya dua proses paralel tidak mengunduh bersamaan.
Mask2FormerForUniversalSegmentation.from_pretrained(PRETRAINED)

SCRIPT = Path("train_mask2former.py").resolve()


def train_cmd(tag, epochs, name):
    return [sys.executable, SCRIPT, "train", "--tag", tag, "--name", name, "--data-root", DATA_ROOT, "--out", RUNS,
            "--weights", PRETRAINED, "--epochs", epochs, "--batch", BATCH, "--lr", LR, "--imgsz", IMGSZ,
            "--workers", WORKERS, "--seed", SEED, "--eval-every", 1 if PILOT else EVAL_EVERY,
            "--max-hours", MAX_HOURS_PER_RUN]


def predict_cmd(tag, name):
    return [sys.executable, SCRIPT, "predict", "--tag", tag, "--name", name, "--data-root", DATA_ROOT,
            "--out", RUNS, "--imgsz", IMGSZ, "--splits", *EVAL_SPLITS]

config.json:   0%|          | 0.00/79.8k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  190MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/554 [00:00<?, ?it/s]

[transformers] Mask2FormerForUniversalSegmentation LOAD REPORT from: facebook/mask2former-swin-tiny-coco-instance
Key                                                                                                                                | Status     | 
-----------------------------------------------------------------------------------------------------------------------------------+------------+-
model.pixel_level_module.encoder.swin.encoder.layers.{0, 1, 2, 3}.blocks.{0, 1, 2, 3, 4, 5}.attention.self.relative_position_index | UNEXPECTED | 
model.pixel_level_module.encoder.swin.layernorm.bias                                                                               | MISSING    | 
model.pixel_level_module.encoder.swin.layernorm.weight                                                                             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were n

In [7]:
import re

LOG_DIR = WORK / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
GPU_LOG = WORK / "gpu_time_log.csv"
N_GPU = torch.cuda.device_count()
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(N_GPU)]
print(f"{N_GPU} GPU:", GPU_NAMES)
_lock = threading.Lock()


def job(kind, name, cmd):
    return {"kind": kind, "name": name, "cmd": [str(c) for c in cmd], "log": str(LOG_DIR / f"{name}_{kind}.log")}


def _last_line(path):
    try:
        with open(path, "rb") as f:
            f.seek(0, 2)
            f.seek(max(0, f.tell() - 4000))
            text = f.read().decode("utf-8", "ignore")
    except FileNotFoundError:
        return ""
    parts = [p.strip() for p in re.split(r"[\r\n]+", text) if p.strip()]
    return parts[-1][:170] if parts else ""


def _run_job(j, gpu):
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
    start, t0 = datetime.now(), time.time()
    with open(j["log"], "w", encoding="utf-8") as log:
        log.write("$ " + " ".join(j["cmd"]) + "\n")
        log.flush()
        rc = subprocess.run(j["cmd"], env=env, stdout=log, stderr=subprocess.STDOUT, cwd=WORK).returncode
    row = {"model": MODEL_KEY, "job": j["name"], "kind": j["kind"], "gpu_index": gpu,
           "gpu_name": GPU_NAMES[gpu] if gpu < N_GPU else "cpu",
           "start": start.isoformat(timespec="seconds"), "end": datetime.now().isoformat(timespec="seconds"),
           "hours": round((time.time() - t0) / 3600, 4), "returncode": rc}
    with _lock:
        pd.DataFrame([row]).to_csv(GPU_LOG, mode="a", header=not GPU_LOG.exists(), index=False)
        status = "OK" if rc == 0 else f"GAGAL (kode {rc}), lihat {j['log']}"
        print(f"[{row['end']}] {j['kind']} {j['name']} di GPU{gpu}: {row['hours']:.2f} jam, {status}", flush=True)
    return rc


def run_jobs(jobs):
    """Dengan 2 GPU: job pertama di GPU0, sisanya berurutan di GPU1 (dua jalur paralel).
    Dengan 1 GPU: semua berurutan di GPU0. Setiap job = satu proses terpisah dengan satu GPU."""
    lanes = {0: jobs[:1], 1: jobs[1:]} if N_GPU >= 2 and len(jobs) > 1 else {0: jobs}
    results, started = {}, set()

    def lane(gpu, lane_jobs):
        for j in lane_jobs:
            started.add(j["name"])
            results[j["name"]] = _run_job(j, gpu)

    threads = [threading.Thread(target=lane, args=(g, js)) for g, js in lanes.items() if js]
    for t in threads:
        t.start()
    t0 = last = time.time()
    while any(t.is_alive() for t in threads):
        time.sleep(2)
        if time.time() - last >= POLL_SECONDS:
            last = time.time()
            for j in jobs:
                if j["name"] in started and j["name"] not in results:
                    print(f"  {(time.time() - t0) / 60:6.1f} mnt | {j['name']}: {_last_line(j['log'])}", flush=True)
    for t in threads:
        t.join()
    return results

2 GPU: ['Tesla T4', 'Tesla T4']


## 5. Training (atau pilot)

In [8]:
order = sorted(TAGS, key=lambda t: (t != "100", t))          # run 100% di jalur GPU0, 025 lalu 050 di GPU1
run_name = lambda tag: f"{MODEL_KEY}_{tag}"

if PILOT:
    name = f"{MODEL_KEY}_100_pilot"
    run_jobs([job("pilot", name, train_cmd("100", PILOT_EPOCHS, name))])
    hours = pd.read_csv(GPU_LOG).query("job == @name")["hours"].iloc[-1]
    per_epoch_min = hours * 60 / PILOT_EPOCHS
    print(f"\nPilot: {PILOT_EPOCHS} epoch dalam {hours * 60:.1f} menit -> ~{per_epoch_min:.1f} menit/epoch "
          f"(termasuk waktu muat model, jadi sedikit lebih besar dari aslinya)")
    print(f"Saran EPOCHS agar run 100% selesai dalam ~{TARGET_HOURS_100} jam: "
          f"{max(1, int(TARGET_HOURS_100 * 60 / per_epoch_min))}")
    print("Langkah berikutnya: isi EPOCHS, set PILOT = False, lalu Save Version -> Save & Run All.")
elif RUN_TRAINING:
    rc = run_jobs([job("train", run_name(t), train_cmd(t, EPOCHS, run_name(t))) for t in order])
    rows = []
    for t in TAGS:
        p = RUNS / run_name(t) / "train_summary.json"
        rows.append(json.loads(p.read_text()) if p.exists() else {"tag": t, "status": "GAGAL, cek log"})
    print(pd.DataFrame(rows).to_string(index=False))
else:
    found = [p for p in glob.glob(PREV_RUNS_GLOB, recursive=True)
             if all((Path(p) / run_name(t)).exists() for t in TAGS)]
    assert found, f"Folder runs berisi {[run_name(t) for t in TAGS]} tidak ditemukan di {PREV_RUNS_GLOB}"
    shutil.copytree(found[0], RUNS, dirs_exist_ok=True)
    print("Memakai bobot dari", found[0])

     2.0 mnt | mask2former_swin_t_100: epoch=1 | lr=0.0000985 | loss=85.63923 | loss_cross_entropy=6.30004 | loss_mask=0.54683 | loss_dice=1.22211 | skipped_iters=0 | train_time_s=99.4 | elapsed_h=0.0276 | gpu
     2.0 mnt | mask2former_swin_t_025: epoch=2 | lr=0.0000660 | loss=89.85378 | loss_cross_entropy=6.64955 | loss_mask=0.42906 | loss_dice=1.30184 | skipped_iters=0 | train_time_s=23.5 | val_segm_AP=0.01446 | 
     4.0 mnt | mask2former_swin_t_100: epoch=2 | lr=0.0000970 | loss=52.63469 | loss_cross_entropy=3.18105 | loss_mask=0.35655 | loss_dice=0.89429 | skipped_iters=0 | train_time_s=91.5 | val_segm_AP=0.11949 | 
     4.0 mnt | mask2former_swin_t_025: epoch=5 | lr=0.0000925 | loss=54.90739 | loss_cross_entropy=3.52405 | loss_mask=0.29358 | loss_dice=0.82921 | skipped_iters=0 | train_time_s=23.8 | elapsed_h=0.0544 | gpu
     6.0 mnt | mask2former_swin_t_100: epoch=3 | lr=0.0000955 | loss=43.0983 | loss_cross_entropy=2.30449 | loss_mask=0.3279 | loss_dice=0.81228 | skipped_iters

## 6. Evaluasi pada valid, test, dan test_own

In [9]:
if not PILOT:
    run_jobs([job("predict", run_name(t), predict_cmd(t, run_name(t))) for t in order
              if (RUNS / run_name(t)).exists()])

    rows, per_class = [], {}
    for t in TAGS:
        summ_p = RUNS / run_name(t) / "train_summary.json"
        summ = json.loads(summ_p.read_text()) if summ_p.exists() else {}
        for s in EVAL_SPLITS:
            p = RUNS / run_name(t) / f"eval_{s}.json"
            if not p.exists():
                continue
            r = json.loads(p.read_text())
            per_class[(t, s)] = r.pop("per_class_segm_AP")
            rows.append({"model": MODEL_LABEL, "family": FAMILY, "fraksi": int(t), "split": s,
                         **{k: round(v, 4) for k, v in r.items() if k.startswith(("segm_", "bbox_"))},
                         "ms_per_image": r.get("ms_per_image"), "train_hours": summ.get("train_hours"),
                         "epochs_done": summ.get("epochs_done"), "stopped_by_time": summ.get("stopped_by_time")})
    metrics = pd.DataFrame(rows)
    metrics.to_csv(WORK / f"metrics_{MODEL_KEY}.csv", index=False)
    pc = pd.DataFrame(per_class).round(4)
    pc.columns = [f"{t}_{s}" for t, s in pc.columns]
    pc.to_csv(WORK / f"per_class_{MODEL_KEY}.csv")

    print("Mask AP (COCO AP@[.50:.95]) per fraksi data latih:")
    print(metrics.pivot(index="fraksi", columns="split", values="segm_AP").reindex(columns=EVAL_SPLITS).to_string(), "\n")
    print("Mask AP50:")
    print(metrics.pivot(index="fraksi", columns="split", values="segm_AP50").reindex(columns=EVAL_SPLITS).to_string(), "\n")
    print("Mask AP per kelas:")
    print(pc.to_string())

     2.0 mnt | mask2former_swin_t_100: [valid] mask AP=0.3324  AP50=0.4329  box AP=0.3371  (16089 prediksi)
     2.0 mnt | mask2former_swin_t_025: [valid] mask AP=0.2709  AP50=0.3663  box AP=0.2658  (16243 prediksi)
[2026-11-15T12:20:01] predict mask2former_swin_t_100 di GPU0: 0.04 jam, GAGAL (kode 1), lihat /kaggle/working/logs/mask2former_swin_t_100_predict.log
[2026-11-15T12:20:02] predict mask2former_swin_t_025 di GPU1: 0.04 jam, GAGAL (kode 1), lihat /kaggle/working/logs/mask2former_swin_t_025_predict.log
     4.0 mnt | mask2former_swin_t_050: [valid] mask AP=0.2823  AP50=0.3902  box AP=0.2833  (16217 prediksi)
[2026-11-15T12:22:20] predict mask2former_swin_t_050 di GPU1: 0.04 jam, GAGAL (kode 1), lihat /kaggle/working/logs/mask2former_swin_t_050_predict.log
Mask AP (COCO AP@[.50:.95]) per fraksi data latih:
split    valid  test
fraksi              
25      0.2709   NaN
50      0.2823   NaN
100     0.3324   NaN 

Mask AP50:
split    valid  test
fraksi              
25      0.3663 

## 7. Ringkasan waktu GPU dan output

In [10]:
if GPU_LOG.exists():
    log = pd.read_csv(GPU_LOG)
    print(log.to_string(index=False))
    print("\nTotal jam-GPU per jenis job:")
    print(log.groupby("kind")["hours"].sum().round(2).to_string())

print("\nOutput penting di", WORK)
for p in sorted(WORK.glob("*.csv")):
    print("  ", p.name)
for p in sorted(RUNS.glob("*/")):
    files = [f.name for f in p.iterdir()]
    print("  ", p.relative_to(WORK), "->", ", ".join(sorted(files))[:150])

             model                    job    kind  gpu_index gpu_name               start                 end  hours  returncode
mask2former_swin_t mask2former_swin_t_025   train          1 Tesla T4 2026-11-15T10:27:28 2026-11-15T11:10:01 0.7093           0
mask2former_swin_t mask2former_swin_t_050   train          1 Tesla T4 2026-11-15T11:10:01 2026-11-15T12:16:06 1.1012           0
mask2former_swin_t mask2former_swin_t_100   train          0 Tesla T4 2026-11-15T10:27:28 2026-11-15T12:17:38 1.8360           0
mask2former_swin_t mask2former_swin_t_100 predict          0 Tesla T4 2026-11-15T12:17:39 2026-11-15T12:20:01 0.0395           1
mask2former_swin_t mask2former_swin_t_025 predict          1 Tesla T4 2026-11-15T12:17:39 2026-11-15T12:20:02 0.0397           1
mask2former_swin_t mask2former_swin_t_050 predict          1 Tesla T4 2026-11-15T12:20:02 2026-11-15T12:22:20 0.0384           1

Total jam-GPU per jenis job:
kind
predict    0.12
train      3.65

Output penting di /kaggle/wor

## Catatan untuk logbook dan laporan

Isi setelah notebook selesai (bagian 4 tugas: bukti proses):

- **Waktu GPU:** salin tabel dari `gpu_time_log.csv` (jam per job, GPU yang dipakai).
- **Log training asli:** `runs/<run>/results.csv` untuk setiap fraksi.
- **Keputusan:** jumlah epoch dan alasannya (hasil pilot), batch size, ukuran input, apakah ada run yang berhenti karena batas waktu (`stopped_by_time`).
- **Kendala:** error, OOM, atau hal yang diubah selama pengerjaan, beserta solusinya.
- **Bantuan AI:** bagian kode yang dibuat dengan AI dan kasus saat saran AI salah atau perlu diperbaiki (bagian 5 tugas).